# TD1 — Maîtrise Statistique des Processus
## Étude du procédé MECANEX
---

### Contexte

L'entreprise **MECANEX** fabrique une pièce dont la caractéristique de qualité est la **dureté** (HB).

| Paramètre | Valeur |
|---|---|
| Taille de sous-groupe | $n = 5$ pièces consécutives |
| Fréquence | 1 prélèvement par heure |
| Nombre de sous-groupes | $m = 20$ |
| Tolérance inférieure $T_i$ | 262 HB |
| Tolérance supérieure $T_s$ | 302 HB |
| Cible $T$ | 282 HB |

---

### Organisation

> **PARTIE A — Mise en œuvre**
> On applique la méthode telle qu'elle se pratique en atelier : **les constantes $A_3$, $B_3$, $B_4$
> sont lues dans la table normalisée et utilisées telles quelles.** On trace les cartes, on applique
> les règles de Nelson, on interprète, on calcule la capabilité, on conclut.
>
> Deux questions de méthode se posent en chemin — elles sont traitées **avec les seuls chiffres de la
> table**, sans théorie :
> - faut-il centrer la carte $\bar X$ sur $\bar{\bar X}$ ou sur la cible ?
> - comment découper la carte $S$ en zones pour appliquer les règles de Nelson ?
>
> **PARTIE B — D'où viennent ces constantes ?** *(séance suivante)*
> On ouvre la boîte noire : pourquoi $A_3 = 1{,}427$, pourquoi $B_3 = 0$, pourquoi 3 écarts-types,
> et ce que vaut vraiment un $C_{pk}$ estimé sur 100 pièces.

**La partie A se fait intégralement sans la partie B.**

# PARTIE A — MISE EN ŒUVRE

## A.0 — Données et table des constantes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from numpy.lib.stride_tricks import sliding_window_view as fenetres

plt.rcParams.update({'figure.figsize': (13, 5), 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3})

# =====================================================================
#  TABLE DES CONSTANTES DE CARTES DE CONTROLE  (extrait d'ISO 7870-2)
#  On les utilise ICI telles quelles. Leur origine : partie B.
#      n  :  (c4,     A3,    B3,    B4)
# =====================================================================
TABLE = {
    2:  (0.7979, 2.659, 0.000, 3.267),
    3:  (0.8862, 1.954, 0.000, 2.568),
    4:  (0.9213, 1.628, 0.000, 2.266),
    5:  (0.9400, 1.427, 0.000, 2.089),
    6:  (0.9515, 1.287, 0.030, 1.970),
    7:  (0.9594, 1.182, 0.118, 1.882),
    8:  (0.9650, 1.099, 0.185, 1.815),
    9:  (0.9693, 1.032, 0.239, 1.761),
    10: (0.9727, 0.975, 0.284, 1.716),
}

print("Table des constantes (ISO 7870-2)\n")
print(f"{'n':>3}{'c4':>9}{'A3':>8}{'B3':>8}{'B4':>8}")
for nn, (c, a, b3, b4) in TABLE.items():
    print(f"{nn:>3}{c:>9.4f}{a:>8.3f}{b3:>8.3f}{b4:>8.3f}")

---

## A.1 — Statistiques par sous-groupe

> **Q1.**  
> - Importer les données du fichier *`data.dat`* dans un tableau *`Numpy`*
> - Calculer, pour chaque sous-groupe $j$, la moyenne $\bar X_j$ et l'écart-type $S_j$.
> - En déduire $\bar{\bar X}$ (moyenne des moyennes) et $\bar S$ (moyenne des écarts-types).
> - Afficher les variables calculées (*`print`*)  
> - Interpréter les valeurs obtenues pour $\bar{\bar X}$ et $\bar S$, que peut on conclure et ne pas conclure ?

> ### ⚠️ Attention au diviseur
>
> $$S_j = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}\big(X_{ij}-\bar X_j\big)^2}$$
>
> En Python : `np.std(..., ddof=1)`. En Excel : `ECARTYPE.STANDARD`, **pas** `ECARTYPE.PEARSON`.
>
> **Les constantes de la table sont définies pour cet estimateur-là.** Les utiliser avec un $S$ calculé
> en $1/n$ donnerait des limites fausses.

---

## A.2 — La carte à la moyenne

**Formules à appliquer** *(constantes lues dans la table)* :

$$\text{LSC} = \text{LC} + A_3\,\bar S \qquad\qquad \text{LIC} = \text{LC} - A_3\,\bar S$$

où LC est la ligne centrale. Avec $n=5$ : $A_3 = 1{,}427$.

> **Q2.**  
> - Calculer les limites avec $\bar{\bar X}$ comme ligne centrale puis avec la cible comme ligne centrale
> - tracer la carte $\bar X$, avec $\bar{\bar X}$ comme ligne centrale puis avec la cible comme ligne centrale
> - Découper les cartes en 6 zones égales afin de préparer le test de Nelson

> ### Le découpage en zones
>
> Les règles de Nelson (§A.4) exigent un découpage de la carte en zones **A**, **B** et **C**, de
> largeur égale au tiers de la distance entre la ligne centrale et la limite :
>
> ```
>    LSC  ────────────────────
>          zone A
>         ────────────────────
>          zone B
>         ────────────────────
>          zone C
>    LC   ════════════════════
>          zone C
>         ────────────────────
>          ...
> ```
>
> Pour la carte $\bar X$, aucune difficulté : la carte est **symétrique** autour de la ligne centrale,
> et les six zones ont exactement la même largeur, $A_3\bar S/3$.
>
> *(On verra en §A.3 que la carte $S$ ne se laisse pas faire aussi facilement.)*

> **Q3.**  
> - Charger les règles de Nelson
> - Appliquer le test aux 2 cartes de contrôle
> - Afficher les résultats dans un tableau pour chacune des règles en indiquant à quels points se déclenchent les alarmes

> ### Les règles de Nelson
>
>La règle de base (« 1 point hors limites ») ne regarde qu'un point à la fois. Les règles
>supplémentaires examinent des **motifs** sur plusieurs points, en s'appuyant sur le découpage en zones.
>
>| N° | Règle | Signature typique |
>|---:|---|---|
>| 1 | 1 point au-delà de 3 zones (hors limites) | Décalage brutal, valeur aberrante |
>| 2 | 9 points consécutifs du même côté de la LC | Décentrage installé |
>| 3 | 6 points consécutifs croissants ou décroissants | Dérive (usure, échauffement) |
>| 4 | 14 points consécutifs en alternance | Sur-réglage |
>| 5 | 2 points sur 3 au-delà de 2 zones, même côté | Décalage marqué |
>| 6 | 4 points sur 5 au-delà de 1 zone, même côté | Petit décalage |
>| 7 | 15 points consécutifs à moins de 1 zone de la LC | Dispersion **trop faible** |
>| 8 | 8 points consécutifs à plus de 1 zone de la LC | Mélange de deux populations |

> **Q4.**  
> - Comparer les différences obtenues (test de Nelson) sur les 2 cartes
> - Quelle interprétation peut on faire pour chacune des cartes

---

## A.3 — La carte à l'écart-type

> **Pourquoi une deuxième carte ?** Les limites de la carte $\bar X$ sont calculées à partir de
> $\bar S$. Si la dispersion du procédé change, ces limites deviennent fausses — sans que la carte
> $\bar X$ ne le signale nécessairement. **Une carte $\bar X$ sans carte de dispersion est
> inexploitable.**

**Formules à appliquer :**

$$\text{LIC}_S = B_3\,\bar S \qquad\qquad \text{LC}_S = \bar S \qquad\qquad \text{LSC}_S = B_4\,\bar S$$

Avec $n=5$ : $B_3 = 0$ et $B_4 = 2{,}089$.

> **Q4.** Calculer les limites de la carte $\bar S$. Que constatez-vous sur la limite inférieure ?

### 💬 Discussion — Comment découper la carte $S$ en zones ?

Pour appliquer les règles de Nelson à la carte $S$, il faut, comme pour la carte $\bar X$, la découper
en zones A / B / C.

**Deux méthodes viennent naturellement à l'esprit.**

**Méthode 1 — découper l'intervalle entre les limites en 6 parts égales.**
C'est ce qu'on a fait pour la carte $\bar X$, et cela semble se transposer directement :
$$\text{largeur d'une zone} = \frac{\text{LSC}_S - \text{LIC}_S}{6} = \frac{(B_4-B_3)\,\bar S}{6}$$

**Méthode 2 — partir de la ligne centrale.**
Une zone fait, par définition, le tiers de la distance entre la ligne centrale et la limite. Comme la
limite **supérieure** n'a pas été touchée par la troncature, on peut s'appuyer dessus :
$$\text{largeur d'une zone} = \frac{\text{LSC}_S - \text{LC}_S}{3} = \frac{(B_4-1)\,\bar S}{3}$$

> **Q5.**  
> - Est ce que l'on obtient le même zonage avec les 2 méthodes ?
> - Quel est le critère à respecter afin que les 2 méthodes donnent le même résultat ?
> - Observe t-on le même problème pour la carte à la moyenne ?

---

## A.6 — Étude de capabilité

> ### ⚠️ L'ordre est contraignant
>
> $$\text{Stabilité} \;\longrightarrow\; \text{Capabilité}$$
>
> **Un indice de capabilité calculé sur un procédé hors contrôle n'a aucun sens.** Il estime les
> paramètres d'une loi qui, par définition, n'est pas stable : le nombre obtenu est bien défini
> arithmétiquement, mais il **ne prédit rien**.
>
> Nous allons quand même le calculer — **précisément pour montrer ce qui cloche**.

### Les formules

Il faut d'abord une estimation de $\sigma$, l'écart-type des **pièces individuelles**. La table fournit
pour cela le coefficient $c_4$ :

$$\hat\sigma_{\text{court terme}} = \frac{\bar S}{c_4} \qquad (c_4 = 0{,}9400 \text{ pour } n=5)$$

*(Pourquoi diviser par $c_4$ ? Voir §B.2. Pour l'instant, on applique la recette.)*

$$C_p = \frac{T_s-T_i}{6\hat\sigma} \qquad\qquad C_{pk} = \min\left(\frac{T_s-\bar{\bar X}}{3\hat\sigma},\ \frac{\bar{\bar X}-T_i}{3\hat\sigma}\right)$$

> **Image.** $C_p$ compare la largeur de la voiture à celle du garage. $C_{pk}$ tient compte de ce que
> la voiture est garée de travers. On a toujours $C_{pk} \le C_p$, avec égalité si le procédé est centré.

Les indices **$P_p$ et $P_{pk}$** utilisent la même formule, mais avec l'écart-type calculé sur
**l'ensemble des 100 données** (dispersion long terme).

| | $C_p$, $C_{pk}$ (**capabilité**) | $P_p$, $P_{pk}$ (**performance**) |
|---|---|---|
| $\hat\sigma$ | $\bar S/c_4$ — dispersion **intra** sous-groupe | Écart-type de **toutes** les données |
| Question | De quoi le procédé est-il **capable** ? | Que **livre**-t-il réellement ? |

> **Q6.**
> - Calculer les quatre indices. **Attention : un seul jeu d'indices pour tout le procédé, surtout pas un par sous-groupe.**
> - Interpréter les résultats

> ### 💬 Le rapport $C_p/P_p$ : l'information la plus riche de l'étude
>
> $$\frac{C_p}{P_p} = \frac{\hat\sigma_{\text{long terme}}}{\hat\sigma_{\text{court terme}}}$$
>
> - Rapport $\approx 1$ : les dispersions court et long terme sont identiques $\Rightarrow$ **procédé
>   stable**. Toute amélioration exigera un investissement sur le système (machine, méthode, matière).
> - Rapport $> 1$ : il existe une variabilité **entre** les sous-groupes $\Rightarrow$ **causes
>   spéciales** présentes.
>
> **L'écart $C_p - P_p$ mesure le gain accessible par la seule maîtrise du procédé, sans le moindre
> investissement.** C'est l'argument économique décisif de la MSP, et le chiffre à mettre en tête d'un
> rapport de direction.

> **Q7.** Calculer ce rapport et l'interpréter.